In [ ]:
path_prefix = path_prefix = "/Users/agobharoun/Documents/IND320/Streamlit_dashboard/streamlit/"

## Reading the data

In [1]:
import pandas as pd

reservoirs = pd.read_csv("../data/reservoirs.csv")
reservoirs.head()

,dato_Id,omrType,omrnr,iso_aar,iso_uke,fyllingsgrad,kapasitet_TWh,fylling_TWh,neste_Publiseringsdato,fyllingsgrad_forrige_uke,endring_fyllingsgrad
0,1995-09-03,EL,4,1995,35,0.944721,21.079208,19.913979,0001-01-01T00:00:00,0.936888,0.007833
1,2020-11-15,EL,1,2020,46,0.974361,6.003264,5.849344,2020-11-25T13:00:00,0.997406,-0.023046
2,2011-08-28,EL,4,2011,34,0.786499,21.079208,16.578772,0001-01-01T00:00:00,0.787193,-0.000694
3,1998-07-05,EL,3,1998,27,0.793969,8.920932,7.082947,0001-01-01T00:00:00,0.733812,0.060157
4,2011-03-27,EL,4,2011,12,0.300820,21.079208,6.341054,0001-01-01T00:00:00,0.311122,-0.010301


In [3]:
print(reservoirs.columns.tolist())

# Display the number of rows and columns.
print("Dataset dimensions:", reservoirs.shape)

# Show each column's data type and number of non-missing values.
reservoirs.info()

['dato_Id', 'omrType', 'omrnr', 'iso_aar', 'iso_uke', 'fyllingsgrad', 'kapasitet_TWh', 'fylling_TWh', 'neste_Publiseringsdato', 'fyllingsgrad_forrige_uke', 'endring_fyllingsgrad']
Dataset dimensions: (14877, 11)
<class 'pandas.DataFrame'>
RangeIndex: 14877 entries, 0 to 14876
Data columns (total 11 columns):
 #   Column                    Non-Null Count  Dtype  
---  ------                    --------------  -----  
 0   dato_Id                   14877 non-null  str    
 1   omrType                   14877 non-null  str    
 2   omrnr                     14877 non-null  int64  
 3   iso_aar                   14877 non-null  int64  
 4   iso_uke                   14877 non-null  int64  
 5   fyllingsgrad              14877 non-null  float64
 6   kapasitet_TWh             14877 non-null  float64
 7   fylling_TWh               14877 non-null  float64
 8   neste_Publiseringsdato    14877 non-null  str    
 9   fyllingsgrad_forrige_uke  14877 non-null  float64
 10  endring_fyllingsgrad     

In [4]:
# Convert the observation date from text to Pandas datetime values.
reservoirs["dato_Id"] = pd.to_datetime(reservoirs["dato_Id"])

# Confirm the conversion and inspect the available date range.
print("Data type:", reservoirs["dato_Id"].dtype)
print("First date:", reservoirs["dato_Id"].min())
print("Last date:", reservoirs["dato_Id"].max())

Data type: datetime64[us]
First date: 1995-01-08 00:00:00
Last date: 2026-09-06 00:00:00


In [5]:
# Inspect examples from the next-publication-date column.
print(reservoirs["neste_Publiseringsdato"].head(10))

# Count the source-system placeholder dates.
placeholder_count = (
    reservoirs["neste_Publiseringsdato"] == "0001-01-01T00:00:00"
).sum()

print("Placeholder dates:", placeholder_count)

0    0001-01-01T00:00:00
1    2020-11-25T13:00:00
2    0001-01-01T00:00:00
3    0001-01-01T00:00:00
4    0001-01-01T00:00:00
5    0001-01-01T00:00:00
6    0001-01-01T00:00:00
7    0001-01-01T00:00:00
8    0001-01-01T00:00:00
9    2023-08-23T13:00:00
Name: neste_Publiseringsdato, dtype: str
Placeholder dates: 11322


In [7]:
# Create a display copy so the original datetime columns remain usable.
reservoirs_display = reservoirs.copy()

# Format the observation date as day.month.year.
reservoirs_display["dato_Id"] = reservoirs_display[
    "dato_Id"
].dt.strftime("%d.%m.%Y")

# Include the time for publication dates and label missing dates clearly.
reservoirs_display["neste_Publiseringsdato"] = reservoirs_display[
    "neste_Publiseringsdato"
].dt.strftime("%d.%m.%Y kl. %H:%M").fillna("Ikke oppgitt")

reservoirs_display.head()

,dato_Id,omrType,omrnr,iso_aar,iso_uke,fyllingsgrad,kapasitet_TWh,fylling_TWh,neste_Publiseringsdato,fyllingsgrad_forrige_uke,endring_fyllingsgrad
0,03.09.1995,EL,4,1995,35,0.944721,21.079208,19.913979,Ikke oppgitt,0.936888,0.007833
1,15.11.2020,EL,1,2020,46,0.974361,6.003264,5.849344,25.11.2020 kl. 13:00,0.997406,-0.023046
2,28.08.2011,EL,4,2011,34,0.786499,21.079208,16.578772,Ikke oppgitt,0.787193,-0.000694
3,05.07.1998,EL,3,1998,27,0.793969,8.920932,7.082947,Ikke oppgitt,0.733812,0.060157
4,27.03.2011,EL,4,2011,12,0.300820,21.079208,6.341054,Ikke oppgitt,0.311122,-0.010301


In [8]:
# Count the observations belonging to each area type.
print("Area types:")
print(reservoirs["omrType"].value_counts())

# Show the available area numbers.
print("\nArea numbers:")
print(sorted(reservoirs["omrnr"].unique()))

# Check how many area observations normally exist for each date.
print("\nRows per observation date:")
print(reservoirs.groupby("dato_Id").size().value_counts().sort_index())

Area types:
omrType
EL      8265
VASS    4959
NO      1653
Name: count, dtype: int64

Area numbers:
[np.int64(0), np.int64(1), np.int64(2), np.int64(3), np.int64(4), np.int64(5)]

Rows per observation date:
9    1653
Name: count, dtype: int64


In [10]:
# Show which area numbers occur within each area type.
area_structure = pd.crosstab(
    reservoirs["omrType"],
    reservoirs["omrnr"]
)

area_structure

# Select only columns stored as numbers.
numeric_columns = reservoirs.select_dtypes(include="number").columns

# Transpose the summary so each original column appears as one row.
numeric_summary = reservoirs[numeric_columns].describe().T

numeric_summary

,count,mean,std,min,25%,50%,75%,max
omrnr,14877.0,2.333333,1.490762,0.000000,1.000000,2.000000,3.000000,5.000000
iso_aar,14877.0,2010.346038,9.146696,1995.000000,2002.000000,2010.000000,2018.000000,2026.000000
iso_uke,14877.0,26.405929,15.017910,1.000000,13.000000,26.000000,39.000000,53.000000
fyllingsgrad,14877.0,0.618852,0.214850,0.055160,0.456234,0.648049,0.802135,1.020072
kapasitet_TWh,14877.0,29.145860,22.739070,6.003264,17.390587,23.237715,34.043590,87.437580
fylling_TWh,14877.0,18.139713,15.966211,0.331142,7.321973,14.501389,22.203530,84.544820
fyllingsgrad_forrige_uke,14877.0,0.618890,0.214843,0.055160,0.456356,0.648114,0.802133,1.020037
endring_fyllingsgrad,14877.0,-0.000038,0.031385,-0.242484,-0.022395,-0.007072,0.016277,0.336623
